<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Bitcoin Foundations
## Chapter 4 – Scripts and Spending Conditions

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


To demystify Script, you first implement a tiny stack-based interpreter that understands a handful of opcodes: pushing data, duplicating the top of the stack, hashing, equality+verify, and a toy signature check. This makes the "locking and unlocking" narrative concrete in a few dozen lines of Python.

In [1]:
from typing import List, Union  # type hints for clarity

Opcode = str  # represent opcodes as strings like "OP_DUP"
Word = Union[bytes, bool]  # stack elements are bytes or booleans


def run_script(script: List[Union[Opcode, Word]], stack: List[Word]) -> List[Word]:
    for token in script:  # walk through script tokens
        if isinstance(token, bytes):  # data push
            stack.append(token)  # push data on stack
        elif token == "OP_DUP":  # duplicate top element
            stack.append(stack[-1])
        elif token == "OP_HASH160":  # toy hash: just mark value
            data = stack.pop()
            stack.append(b"H(" + data + b")")  # stand-in for real hash160
        elif token == "OP_EQUALVERIFY":  # equality check with pop
            a = stack.pop()
            b = stack.pop()
            if a != b:
                raise ValueError("OP_EQUALVERIFY failed")
        elif token == "OP_CHECKSIG":  # toy signature check
            stack.pop()  # pretend to consume signature
            stack.pop()  # pretend to consume public key
            stack.append(True)  # push success flag
        else:
            raise ValueError(f"Unknown opcode: {token}")
    return stack  # final stack state


With the interpreter in place, you can simulate a simplified P2PKH spend by concatenating a toy `scriptSig` and `scriptPubKey`. Watching how the stack evolves and ends in `[True]` makes the spend condition feel much more mechanical and less like black magic.

In [2]:
sig = b"<sig>"  # placeholder for signature bytes
pub = b"<pubkey>"  # placeholder for public key bytes
expected_hash = b"H(" + pub + b")"  # toy hash160(pubkey)

script_sig = [sig, pub]  # unlocking script pushes sig and pubkey
script_pubkey = [  # locking script template
    "OP_DUP",
    "OP_HASH160",
    expected_hash,
    "OP_EQUALVERIFY",
    "OP_CHECKSIG",
]

stack: List[Word] = []  # start with empty stack
stack = run_script(script_sig + script_pubkey, stack)  # run combined script

print("Final stack:", stack)  # should end with [True] in this toy model


Final stack: [True]


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
